# VoteLens: document-grounded claim verification
## HackApertus Track 2A — editable demonstration with saved results

VoteLens classifies claims against supplied Swiss voting sources as entailment (0), neutral (1) or contradiction (2). It supports German, French and Italian. The notebook calls the Docker application using CSCS Apertus v1.5 70B.

**Saved execution:** the main cells retain the observed **15 October 2026 acceptance-claim run** of the corrected candidate: both Task A and B returned contradiction and exited zero. The claim says the Federal Council recommends accepting the initiative; the fixed reference and page-19 PDF evidence recommend rejection.

**The appendix restores the earlier rejection-claim run**, where both tasks returned entailment. The code fingerprints differ between those recorded runs, so they should not be treated as a controlled comparison of claim wording alone.

Live execution is disabled for sharing. Saved outputs remain visible without API access. To try a new input, enable live mode and rerun the setup and execution cells; previously saved outputs are not new results for edited inputs.


## 1. Choose a claim and source

Edit the next cell to explore different claims. **Changing a claim does not change the reference passage.** After editing, rerun the setup cell and execution cells below; old results are not results for a newly edited claim.

### Booklet catalogue
| String for `BOOKLET_KEY` | Source language | Description |
|---|---|---|
| `sustainability_fr_2026` | French | Federal Chancellery explanations for the 14 June 2026 vote. The selected proposal is “No to a Switzerland of 10 million” (sustainability initiative), concerning population growth and immigration. The default claim concerns the Federal Council’s recommendation. The earlier smoke found supporting text on physical PDF page 19. |

One booklet is provided with observed execution evidence. Add another entry to `BOOKLETS` using a Federal Chancellery PDF, its source language, and the exact proposal name; do not invent a booklet URL.

### Claims to try
| String for `CLAIM_KEY` | Expected relation to the supplied reference | Claim |
|---|---|---|
| `reject_fr` | Entailment | The Federal Council recommends rejecting the initiative. |
| `accept_fr` | Contradiction | The Federal Council recommends accepting the initiative. |
| `neutral_fr` | Neutral — **illustrative only; not a verified successful Neutral run** | The vote will take place on a Monday. |

The rejection and acceptance claims have historical saved outputs below. `neutral_fr` is a labelled teaching example, not a verified successful Neutral execution. A model prediction is never guaranteed.

For Task B, `REFERENCE_KEY="council_position"` uses the fixed reference from the previous smoke. `unrelated_example` is an explicitly **authored toy reference** about a library opening time; it is not a quotation from a voting booklet. Edit `CLAIM_TEXT` and `REFERENCE_TEXT` independently for arbitrary text.


In [12]:
# EDIT THIS CELL, then run the remaining cells in order.
BOOKLET_KEY = "sustainability_fr_2026"  # Options described above.
CLAIM_KEY = "accept_fr"                # reject_fr / accept_fr / neutral_fr.
CLAIM_TEXT = ""                        # Nonempty text overrides CLAIM_KEY.
CLAIM_LANGUAGE = "fr"                  # de / fr / it for CLAIM_TEXT.
REFERENCE_KEY = "council_position"      # Or unrelated_example.
REFERENCE_TEXT = ""                    # Independent from CLAIM_TEXT.
REFERENCE_LANGUAGE = "fr"              # de / fr / it for REFERENCE_TEXT.
RUN_TASKS = ["B", "A"]                 # ["B"] avoids PDF/retrieval work.
LIVE_RUN = False                       # True enables paid endpoint calls.


## 2. Reproduce the repository layout

Repository: **`greateden/Hackathon26`**, branch **`main`**. The notebook is under `track_2a/docs/`:

```text
Hackathon26/
├── Makefile
└── track_2a/
    ├── Makefile
    ├── docs/HackApertus-demo.ipynb
    └── src/
```

With repository access:
```bash
git clone --branch main https://github.com/greateden/Hackathon26.git
cd Hackathon26
python -m jupyter lab
```

Open this notebook from the checkout root or from `track_2a/docs/`. Setup searches the current directory and its parents for `track_2a/compose.submission.yaml`; it does not select a separate candidate folder.

Docker Desktop/Engine and Compose, GNU Make, Jupyter and IPython are required. Application dependencies and the bundled encoder come from the Docker image. Supply the key privately through `LLM_API_KEY`, `CSCS_INFERENCE_API_KEY`, or the hidden prompt. Set `LIVE_RUN=True` above and run cells in order. Each selected task uses API resources.

Saved results below retain their original source fingerprints. They do not automatically verify later commits on `main`; rerun to measure the checked-out version.


In [13]:
from pathlib import Path
import json, os, subprocess, uuid, hashlib, shutil
from html import escape
from IPython.display import display, HTML

# The notebook lives in track_2a/docs; set VOTELENS_PROJECT_ROOT only for unusual launches.
def find_project():
    configured = os.environ.get("VOTELENS_PROJECT_ROOT")
    bases = [Path(configured).expanduser()] if configured else [Path.cwd(), *Path.cwd().parents]
    candidates = bases
    for candidate in candidates:
        if (candidate / "track_2a" / "compose.submission.yaml").is_file():
            return candidate.resolve()
    raise RuntimeError("Start Jupyter in the checkout or track_2a/docs, or set VOTELENS_PROJECT_ROOT.")

PROJECT = find_project()
MODEL = "swiss-ai/Apertus-v1.5-70B"
BASE_URL = os.environ.get("BASE_URL", "https://api.inference.cscs.ch/v1")
BOOKLETS = {
    "sustainability_fr_2026": {
        "path": "https://www.bk.admin.ch/dam/fr/sd-web/WeUrKyC0FyPc/2026-06-14_explications_du_conseil_federal.pdf",
        "language": "fr",
        "vote": "Initiative populaire « Pas de Suisse à 10 millions ! (initiative pour la durabilité) »",
    }
}
CLAIMS = {
    "reject_fr": ("Le Conseil fédéral recommande de rejeter l’initiative pour la durabilité.", "fr"),
    "accept_fr": ("Le Conseil fédéral recommande d’accepter l’initiative pour la durabilité.", "fr"),
    "neutral_fr": ("La votation aura lieu un lundi.", "fr"),
}
REFERENCES = {
    "council_position": ("Le Conseil fédéral recommande de rejeter l’initiative pour la durabilité.", "fr"),
    "unrelated_example": ("La bibliothèque ouvre à neuf heures le lundi.", "fr"),
}
source = BOOKLETS[BOOKLET_KEY]
claim, claim_language = ((CLAIM_TEXT, CLAIM_LANGUAGE) if CLAIM_TEXT.strip() else CLAIMS[CLAIM_KEY])
reference, reference_language = ((REFERENCE_TEXT, REFERENCE_LANGUAGE) if REFERENCE_TEXT.strip() else REFERENCES[REFERENCE_KEY])
assert claim_language in ("de", "fr", "it") and reference_language in ("de", "fr", "it")
assert RUN_TASKS and all(t in ("A", "B") for t in RUN_TASKS)
REQUEST_B = {"id":"notebook-task-B", "vote":source["vote"],
             "reference":{"text":reference,"language":reference_language},
             "claim":{"text":claim,"language":claim_language}}
REQUEST_A = {"id":"notebook-task-A", "vote":source["vote"],
             "booklet":{"path":source["path"],"language":source["language"]},
             "claim":{"text":claim,"language":claim_language}}
print("Selected claim:", claim)
print("Task B reference:", reference)
print("Booklet:", BOOKLET_KEY, "| Tasks:", RUN_TASKS)

def show_result(result, origin):
    display(HTML(f"<h3>{escape(origin)}: {escape(result['label_name'])}</h3>"))
    for item in result["evidence"]:
        page = "Supplied reference" if item["page"] is None else f"PDF page {item['page']}"
        display(HTML(f"<p><b>{escape(page)}</b></p><blockquote>{escape(item['text'])}</blockquote>"))
    rows = "".join(f"<tr><td>{escape(k)}</td><td>{v}</td></tr>" for k,v in result["metrics"].items())
    display(HTML(f"<table><tr><th>Metric</th><th>Value</th></tr>{rows}</table>"))

SOURCE_HASHES = {name: hashlib.sha256((PROJECT / name).read_bytes()).hexdigest()
                 for name in ["track_2a/src/booklet_app.py", "track_2a/compose.submission.yaml"]}
print("Pipeline: vote-aware-v1; endpoint model:", MODEL)
print("Source fingerprints:", json.dumps(SOURCE_HASHES, indent=2))


Selected claim: Le Conseil fédéral recommande d’accepter l’initiative pour la durabilité.
Task B reference: Le Conseil fédéral recommande de rejeter l’initiative pour la durabilité.
Booklet: sustainability_fr_2026 | Tasks: ['B', 'A']
Pipeline: vote-aware-v1; endpoint model: swiss-ai/Apertus-v1.5-70B
Source fingerprints: {
  "track_2a/src/booklet_app.py": "27b0018a5523f77828ce69ffda196659a095030fbc9b56f50267594f26e33277",
  "track_2a/compose.submission.yaml": "2cb705b37b52a332404181b5717ac8dca9b3d108344343c6eab8688a7f76b4df"
}


## Task B — inspect the selected reference and claim

The reference remains independent of the edited claim. Compare the meanings to interpret the prediction. Task B has no physical PDF page. The next cell displays the exact request.


In [14]:
print(json.dumps(REQUEST_B, ensure_ascii=False, indent=2))


{
  "id": "notebook-task-B",
  "vote": "Initiative populaire « Pas de Suisse à 10 millions ! (initiative pour la durabilité) »",
  "reference": {
    "text": "Le Conseil fédéral recommande de rejeter l’initiative pour la durabilité.",
    "language": "fr"
  },
  "claim": {
    "text": "Le Conseil fédéral recommande d’accepter l’initiative pour la durabilité.",
    "language": "fr"
  }
}


## Task A — inspect the selected booklet and claim

The next request identifies the booklet and proposal. SentenceSwissBERT retrieves passages using the proposal name and claim; Apertus resolves the vote and selects evidence IDs. The PDF URL is for this interactive demo; the evaluation batch CLI instead requires supplied local PDFs.


In [15]:
print(json.dumps(REQUEST_A, ensure_ascii=False, indent=2))


{
  "id": "notebook-task-A",
  "vote": "Initiative populaire « Pas de Suisse à 10 millions ! (initiative pour la durabilité) »",
  "booklet": {
    "path": "https://www.bk.admin.ch/dam/fr/sd-web/WeUrKyC0FyPc/2026-06-14_explications_du_conseil_federal.pdf",
    "language": "fr"
  },
  "claim": {
    "text": "Le Conseil fédéral recommande d’accepter l’initiative pour la durabilité.",
    "language": "fr"
  }
}


## Credential and Docker execution

Credentials are supplied privately and are never printed. Each run writes uniquely named input, result and diagnostic files under `track_2a/data/notebook-demo/`, leaving the committed `request.json` unchanged. Full diagnostics are not displayed in this notebook. A failed run raises an error; it is never replaced by a stored success or a guessed label.


In [17]:
if LIVE_RUN and not (os.environ.get("LLM_API_KEY") or os.environ.get("CSCS_INFERENCE_API_KEY")):
    from getpass import getpass
    os.environ["LLM_API_KEY"] = getpass("Inference API key (hidden): ").strip()


In [18]:
def run_docker(request, task):
    if not LIVE_RUN:
        print("Live inference disabled. Set LIVE_RUN=True to use API credit.")
        return None
    key = (os.environ.get("LLM_API_KEY") or os.environ.get("CSCS_INFERENCE_API_KEY") or "").strip()
    if not key:
        raise RuntimeError("Set LLM_API_KEY privately before running.")
    if not (PROJECT / "track_2a" / "Makefile").is_file():
        raise RuntimeError("Update PROJECT to the repository root.")
    run_id = uuid.uuid4().hex
    folder = PROJECT / "track_2a" / "data" / "notebook-demo" / run_id
    folder.mkdir(parents=True)
    (folder / "request.json").write_text(json.dumps(request, ensure_ascii=False), encoding="utf-8")
    env = os.environ.copy()
    env["LLM_API_KEY"] = key
    env.update(LLM_NAME=MODEL, LLM_BASE_URL=BASE_URL, LLM_TOKEN_PREFLIGHT="optional",
               REQUEST_FILE=f"/app/data/notebook-demo/{run_id}/request.json",
               RESULT_FILE=f"/app/data/notebook-demo/{run_id}/result.json",
               DIAGNOSTICS_FILE=f"/app/data/notebook-demo/{run_id}/diagnostics.json")
    print(f"Running repository-root make run for Task {task}...")
    process = subprocess.run(["make", "-C", "track_2a", "run"], cwd=PROJECT, env=env,
                             capture_output=True, text=True)
    # Do not display raw environment or diagnostics; they are unnecessary for the demo.
    if process.returncode:
        key = env["LLM_API_KEY"]
        log = (process.stdout + process.stderr).replace(key, "[REDACTED]")
        (folder / "failure.log").write_text(log, encoding="utf-8")
        raise RuntimeError(f"make run failed (exit {process.returncode}); inspect {folder / 'failure.log'} privately.")
    result = json.loads((folder / "result.json").read_text(encoding="utf-8"))
    assert set(result) == {"id", "label", "label_name", "evidence", "metrics"}
    assert result["id"] == request["id"]
    assert type(result["label"]) is int and result["label"] in (0,1,2)
    assert all(type(v) is int and v >= 0 for v in result["metrics"].values())
    (folder / "source-fingerprints.json").write_text(json.dumps(SOURCE_HASHES, indent=2), encoding="utf-8")
    print("make run completed with exit code 0.")
    show_result(result, f"Live Task {task}")
    return result


## Execute selected tasks

This cell runs the selected inputs through the existing Docker application. A failed call raises an error rather than displaying a recorded success. The summary reports predictions without assuming edited claims have the default example’s label.


In [19]:
RESULTS = {}
if LIVE_RUN:
    for task in RUN_TASKS:
        request = {"B": REQUEST_B, "A": REQUEST_A}[task]
        RESULTS[task] = run_docker(request, task)
else:
    print("Not executed: enable LIVE_RUN to generate current-version outputs.")


Running repository-root make run for Task B...
make run completed with exit code 0.


Metric,Value
input_tokens,519
output_tokens,75
inference_time_ms,1187


Running repository-root make run for Task A...
make run completed with exit code 0.


Metric,Value
input_tokens,1316
output_tokens,76
inference_time_ms,1769


In [20]:
if RESULTS:
    rows = ""
    for task, result in RESULTS.items():
        metrics = result["metrics"]
        pages = ", ".join(str(e["page"]) for e in result["evidence"] if e["page"] is not None) or "Reference / no PDF page"
        values = [task, result["label_name"], pages,
                  metrics["input_tokens"], metrics["output_tokens"], metrics["inference_time_ms"]]
        rows += "<tr>" + "".join("<td>" + escape(str(v)) + "</td>" for v in values) + "</tr>"
    headers = ["Task", "Prediction", "Evidence page", "Input tokens", "Output tokens", "Inference ms"]
    display(HTML("<h3>Current execution results</h3><table><tr>" + "".join("<th>"+h+"</th>" for h in headers) + "</tr>" + rows + "</table>"))
    print("Model:", MODEL, "| Pipeline: vote-aware-v1")
else:
    print("No current results available. No benchmark or execution success is claimed.")


Task,Prediction,Evidence page,Input tokens,Output tokens,Inference ms
B,contradiction,Reference / no PDF page,519,75,1187
A,contradiction,19,1316,76,1769


Model: swiss-ai/Apertus-v1.5-70B | Pipeline: vote-aware-v1


## Interpretation and limitations

- These are two smoke examples, not an accuracy benchmark or proof of cross-language generalisation.
- Evidence IDs and text checks establish provenance within supplied passages; reviewers should also inspect whether the evidence supports the prediction.
- Retrieval can omit decisive information. Fixed-size passages may split sentences; scanned PDFs need OCR, which is not implemented.
- This notebook deliberately uses the separate one-request `make run` demonstration path. The production image has a mixed-JSONL entrypoint; these saved notebook calls do not establish that batch path.
- Inference time is the API completion latency, excluding Docker startup, PDF extraction, retrieval and initial downloads.
- Source fingerprints identify the local application files. They do not attest the remote model’s underlying weights or establish byte-for-byte cross-platform reproducibility.

For architecture, development evaluation and recorded experimental limitations, consult [`../technical_report.md`](../technical_report.md) and [`P0_CORRECTNESS.md`](P0_CORRECTNESS.md). Historical experiments used different pipelines; their results should not be attributed to these notebook runs.




## Appendix: earlier rejection-claim run

These outputs are from the **earlier 15 October 2026 run**, not the acceptance claim selected above. The French claim and Task B reference both recommended rejecting the sustainability initiative. Both tasks returned entailment and exited zero. Task B originally had no vote field.

Task B: 519 input tokens, 71 output tokens, 1,096 ms. Task A: 1,345 input tokens, 93 output tokens, 1,524 ms; page 19. These are recorded observations restored from the earlier saved notebook, with no new inference. They used different source fingerprints from the corrected candidate run above.


In [ ]:
# Preserved earlier rejection-claim outputs; no API call is made by this cell.


Running repository-root make run for Task B...
make run completed with exit code 0.


Metric,Value
input_tokens,519
output_tokens,71
inference_time_ms,1096


Running repository-root make run for Task A...
make run completed with exit code 0.


Metric,Value
input_tokens,1345
output_tokens,93
inference_time_ms,1524


Task,Prediction,Matches expected,Evidence page,Input tokens,Output tokens,Inference ms
B,entailment,True,Reference / no PDF page,519,71,1096
A,entailment,True,19,1345,93,1524


Model: swiss-ai/Apertus-v1.5-70B | Pipeline: vote-aware-v1


## Saved run summary

| Recorded claim | Task | Result | Evidence | Input tokens | Output tokens | Inference ms |
|---|---|---|---|---:|---:|---:|
| Council recommends rejection (earlier run) | B | Entailment | Supplied reference | 519 | 71 | 1,096 |
| Council recommends rejection (earlier run) | A | Entailment | PDF page 19 | 1,345 | 93 | 1,524 |
| Council recommends acceptance (current candidate) | B | Contradiction | Supplied reference | 519 | 75 | 1,187 |
| Council recommends acceptance (current candidate) | A | Contradiction | PDF page 19 | 1,316 | 76 | 1,769 |

All four calls completed through the Docker application. These smoke examples are not an accuracy benchmark. Inference time excludes Docker build/startup and retrieval.
